# Exercise: House price levels and dispersion

For this exercise, we're using data on around 1,500 observations of house prices and house characteristics from Ames, a small city in Iowa.

1. Load the Ames housing data set from `ames_houses.csv` located in the `data/` folder. 
2. Restrict the data to the columns `SalePrice` and `Neighborhood`.
3. Check that there are no observations with missing values in this data.
4. Compute the average house price (column `SalePrice`) by neighborhood (column `Neighborhood`). List the three most expensive neighborhoods, for example by using [`sort_values()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html).
5. You are interested to quantify the price dispersion in each neighborhood. To this end, compute the standard deviation by neighborhood using [`std()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html). Which are the three neighborhoods with the most dispersed prices?
6. An alternative measure of dispersion is the ratio of the 90th and 10th percentile of the house price distribution. Use the [`quantile()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html) method to compute the P90 and P10 statistics by neighborhood, compute their ratio and print the three neighborhoods with the largest dispersion.

    *Hint:* The `quantile()` function takes _quantiles_ as arguments, i.e., instead of the 90th percentile you need to specify the quantile as 0.9.

In [263]:
import pandas as pd
import numpy as np

DATA_PATH = '/Users/andreas/NHH/Tech2-local/TECH2-H24/data'
df = pd.read_csv(f'{DATA_PATH}/ames_houses.csv')

In [264]:
# 2, 3
df = df[['SalePrice', 'Neighborhood']].copy()
df.isna().sum()

SalePrice       0
Neighborhood    0
dtype: int64

In [265]:
# 4
df.groupby('Neighborhood')['SalePrice'].mean().sort_values().tail(3)

# Alternative way:
#df.groupby('Neighborhood')['SalePrice'].agg('mean').sort_values().tail(3)

Neighborhood
StoneBr    310499.000000
NridgHt    316270.623377
NoRidge    335295.317073
Name: SalePrice, dtype: float64

In [266]:
# 5
#df.groupby('Neighborhood')['SalePrice'].agg(['min','max','median'])

df.groupby('Neighborhood')['SalePrice'].std().sort_values(ascending = False).head(3)

Neighborhood
NoRidge    121412.658640
StoneBr    112969.676640
NridgHt     96392.544954
Name: SalePrice, dtype: float64

In [267]:
# 6
quant = df.groupby('Neighborhood')['SalePrice'].agg(
    P10 = lambda x: x.quantile(0.10),
    P90 = lambda x: x.quantile(0.90),
)
quant['Dispersion'] = (quant['P90']/ quant['P10'])
quant['Dispersion'].sort_values(ascending=False).head(3)

Neighborhood
IDOTRR     2.546182
StoneBr    2.533834
BrkSide    2.309796
Name: Dispersion, dtype: float64

***
# Exercise: Determinants of house prices

For this exercise, we're using data on around 1,500 observations of house prices and house characteristics from Ames, a small city in Iowa.

1.  Load the Ames housing data set from `ames_houses.csv` located in the `data/` folder. 
2.  Restrict the data to the columns `SalePrice`, `LotArea` and `Bedrooms`.
3.  Restrict your data set to houses with one or more bedrooms and a lot area of at least 100m².
4.  Compute the average lot area. Create a new column `LargeLot` which takes on the value of 1 if the lot area is above the average (_"large"_), and 0 otherwise (_"small"_). 

    What is the average lot area within these two categories?
5.  Create a new column `Rooms` which categorizes the number of `Bedrooms` into three groups: 1, 2, and 3 or more. You can create these categories using boolean indexing, [`np.where()`](https://numpy.org/doc/2.0/reference/generated/numpy.where.html), pandas's [`where()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.where.html), or some other way.
6.  Compute the mean `SalePrice` within each group formed by `LargeLot` and `Rooms` (for a total of 6 different categories) using [`groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html).
7.  Compute and report the average price difference between 1 and 2 bedrooms for a house with a small lot area.
8.  Compute and report the average price difference between a small and a large lot for a house with 2 bedrooms.

In [268]:
# 1, 2, 3
df = pd.read_csv(f'{DATA_PATH}/ames_houses.csv')
df = df[['SalePrice', 'LotArea', 'Bedrooms']]
df = df.query('Bedrooms >= 1 & LotArea >= 100')

In [269]:
# 4
mean = df['LotArea'].mean()
df['Largelot'] = np.where(df['LotArea'] > mean, 1, 0)
df.head(5)


,SalePrice,LotArea,Bedrooms,Largelot
0,208500.0,784.954075,3,0
1,181500.0,891.782144,3,0
2,223500.0,1045.057200,3,1
3,140000.0,887.137445,3,0
4,250000.0,1324.668060,4,1


In [270]:
# 5, 6
df['Rooms'] = np.where(df['Bedrooms'] == 1, 1, np.where(df['Bedrooms'] == 2, 2, 3))
new = df.groupby(['Rooms', 'Largelot'])[['SalePrice']].mean()
new

SalePrice
Rooms Largelot               
1     0         135182.388889
      1         270825.357143
2     0         144739.841379
      1         215591.294118
3     0         164034.885572
      1         222596.090293

In [271]:
# 7, 8
Price1 = df.query('Largelot == 0 & Rooms == 1').mean()
Price2 = df.query('Largelot == 0 & Rooms == 2').mean()
print(f' Average price difference between a small lot house with two or one bedrooms Is : {Price2['SalePrice'] - Price1['SalePrice']}')

print(f' Average price difference between small and large lot house with two bedrooms is: {df.query('Largelot == 1 & Rooms == 2')['SalePrice'].mean() - df.query('Largelot == 0 & Rooms == 2')['SalePrice'].mean()}')

 Average price difference between a small lot house with two or one bedrooms Is : 9557.452490421478
 Average price difference between small and large lot house with two bedrooms is: 70851.4527383367


***
# Exercise: Inflation and unemployment in the US

In this exercise, you'll be working with selected macroeconomic variables for the United States reported at monthly frequency obtained from [FRED](https://fred.stlouisfed.org/).
The data set starts in 1948 and contains observations for a total of 864 months.

1.  Load the data from the file `FRED_monthly.csv` located in the `data/` folder. Print the first 10 observations to get an idea how the data looks like.
2.  Keep only the columns `Year`, `Month`, `CPI`, and `UNRATE`. Moreover, perform this analysis only on observations prior to 1970 and drop the rest.
3.  Since pandas has great support for time series data, we want to create an index based on observation dates. 

    -   To this end, use [`to_datetime()`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) to convert the `Year` and `Month` columns into a date.

        *Hint:* `to_datetime()` requires information on Year/Month/Day, so you need to create a `Day` column first and assign it a value of 1.
        You can then call `to_datetime()` with the argument `df[['Year', 'Month', 'Day']]` to create the corresponding date.
    -   Store the date information in the column `Date`. Delete the columns `Year`, `Month` and `Day` once you are done as these are no longer needed.
    -   Set the `Date` column as the index for the `DataFrame` using [`set_index()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html).

4.  The column `CPI` stores the consumer price index for the US. You may be more familiar with the concept of inflation, which is the percent change of the CPI relative to the previous period. 
    Create a new column `Inflation` which contains the _annual_ inflation _in percent_ relative to the same month in the previous year by applying 
    [`pct_change()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pct_change.html) to the column `CPI`.

    *Hints:* 
    
    -   Since this is monthly data, you need to pass the arguments `periods=12` to `pct_change()` to get annual percent changes.
    -   You need to multiply the values returned by `pct_change()` by 100 to get percent values.

5.  Compute the average unemployment rate (column `UNRATE`) over the whole sample period. Create a new column `UNRATE_HIGH` that contains an indicator whenever the unemployment rate is above its average value (_"high unemployment period"_). 
    -   How many observations fall into the high- and the low-unemployment periods?
    -   What is the average unemployment rate in the high- and low-unemployment periods?
6.  Compute the average inflation rate for high- and low-unemployment periods. Is there any difference?
7.  Use [`resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) to aggregate
    the inflation data to annual frequency and compute the average inflation within each calendar year.

    Which are the three years with the highest inflation rates in the sample?

    *Hint:* Use the resampling rule `'YE'` when calling `resample()`.


In [272]:
# 1, 2
df = pd.read_csv(f'{DATA_PATH}/FRED_monthly.csv')
df = df[['Year','Month', 'CPI', 'UNRATE']]
df = df.query('Year < 1970')

In [273]:
# 3
df['Day'] = 1
df['Date'] = pd.to_datetime(df[['Year','Month','Day']])
df = df.drop(['Year', 'Month', 'Day'], axis = 'columns')
df.set_index('Date',inplace = True)
df

,CPI,UNRATE
Date,,
1948-01-01,23.7,3.4
1948-02-01,23.7,3.8
1948-03-01,23.5,4.0
1948-04-01,23.8,3.9
1948-05-01,24.0,3.5
...,...,...
1969-08-01,36.9,3.5
1969-09-01,37.1,3.7
1969-10-01,37.3,3.7


In [274]:
# 4
df['Inflation'] = (df['CPI'].pct_change(periods = 12))*100
df.head(14)

,CPI,UNRATE,Inflation
Date,,,
1948-01-01,23.7,3.4,NaN
1948-02-01,23.7,3.8,NaN
1948-03-01,23.5,4.0,NaN
1948-04-01,23.8,3.9,NaN
1948-05-01,24.0,3.5,NaN
1948-06-01,24.2,3.6,NaN
1948-07-01,24.4,3.6,NaN
1948-08-01,24.4,3.9,NaN
1948-09-01,24.4,3.8,NaN


In [275]:
# 5
mean = df['UNRATE'].mean()
df['UNRATE_HIGH'] = np.where(df['UNRATE'] > mean, 1, 0)
High = df.query('UNRATE_HIGH == 1').count()['UNRATE_HIGH']
Low = df.query('UNRATE_HIGH == 0').count()['UNRATE_HIGH']
print(High, Low)
High = df.query('UNRATE_HIGH == 1').mean()['UNRATE']
Low = df.query('UNRATE_HIGH == 0').mean()['UNRATE']
print(High, Low)


123 141
5.78130081300813 3.6978723404255316


In [276]:
# 5 Better way to solve
df['UNRATE_HIGH'].value_counts()

UNRATE_HIGH
0    141
1    123
Name: count, dtype: int64

In [280]:
df.groupby('UNRATE_HIGH')['UNRATE'].mean()

UNRATE_HIGH
0    3.697872
1    5.781301
Name: UNRATE, dtype: float64

In [282]:
df.groupby('UNRATE_HIGH')['Inflation'].mean()

UNRATE_HIGH
0    3.110456
1    0.942056
Name: Inflation, dtype: float64

In [290]:
df['Inflation'].resample('YE').mean().sort_values(ascending=False).head(3)
df.resample('YE')['Inflation'].mean().sort_values(ascending=False).head(3)

Date
1951-12-31    7.987456
1969-12-31    5.432647
1968-12-31    4.241319
Name: Inflation, dtype: float64